# ⚙️ Preprocesamiento Avanzado — CAP Sleep Database

**Base de datos:** CAP Sleep Database (PhysioNet)  
**Prerequisito:** Este notebook se ejecuta **después** de `analisis_completo_cap_sleep.ipynb`.

---

## Objetivo

Aplicar un pipeline de preprocesamiento completo sobre las señales EEG de la CAP Sleep Database, preparándolas para la extracción de características y clasificación supervisada de trastornos del sueño.

## ¿Por qué es crítico el preprocesamiento en EEG?

Las señales electroencefalográficas (EEG) son particularmente sensibles al ruido y a las condiciones de adquisición. A diferencia de datos tabulares convencionales, presentan desafíos específicos:

- **Ruido biológico**: artefactos musculares (EMG), movimientos oculares (EOG), actividad cardíaca (ECG) que se superponen a la señal cerebral.
- **Ruido técnico**: impedancia de electrodos, interferencia de línea eléctrica (50/60 Hz), saturación del amplificador (clipping en ±1000 µV).
- **Heterogeneidad multicéntrica**: la CAP Sleep Database fue recopilada en múltiples centros de sueño, cada uno con su propia convención de nomenclatura de canales, frecuencia de muestreo y configuración de amplificadores.
- **Desbalance de clases**: NFLE concentra el 37% de los pacientes mientras que Bruxismo solo el 1.9%, lo que sesga cualquier clasificador no ponderado.

Un preprocesamiento inadecuado contamina las características extraídas y degrada directamente el desempeño de los modelos. Por ejemplo, si no se normaliza la frecuencia de muestreo, las características espectrales (potencia en bandas alfa, beta, etc.) no son comparables entre pacientes.

## Pipeline implementado

| Paso | Sección | Descripción |
|:---:|---|---|
| 1 | §4 | Exclusión de pacientes n13 y n14 (saturación completa) |
| 2 | §5 | Resampleo a 256 Hz |
| 3 | §6 | Normalización de nomenclatura de canales |
| 4 | §7 | Selección de canales relevantes (EEG, EMG, ECG) |
| 5 | §8 | Escalamiento por canal (StandardScaler) |
| 6 | §9 | Normalización por tiempo de sueño efectivo |
| 7 | §10 | Balanceo de clases (SMOTE, ponderación) |
| 8 | §11 | Exportación de datasets preprocesados |


## 2. Importaciones y configuración


In [ ]:
import re
import os
import pickle
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import signal as scipy_signal
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.utils.class_weight import compute_class_weight

# SMOTE requiere imbalanced-learn: pip install imbalanced-learn
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print('✅ imbalanced-learn disponible')
except ImportError:
    SMOTE_AVAILABLE = False
    print('⚠️  imbalanced-learn no instalado. Instalar con: pip install imbalanced-learn')
    print('   El balanceo con SMOTE se omitirá; se usará ponderación de clases.')

mne.set_log_level('ERROR')
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'figure.figsize': (12, 5),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})
sns.set_style('whitegrid')

# ── Rutas ────────────────────────────────────────────────────
BASE_PATH    = Path.cwd() / '../data/cap-sleep-database'
OUT_DIR      = BASE_PATH.parent / 'diagnostico_outputs'
PROCESSED_DIR = BASE_PATH.parent / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

print(f'Base de datos : {BASE_PATH.resolve()}')
print(f'Salida        : {PROCESSED_DIR.resolve()}')


## 3. Carga de datos

Reutilizamos las constantes y el parser del notebook de análisis, y cargamos las señales EDF de los pacientes representativos. Para el preprocesamiento completo, procesamos **todos los archivos EDF** de forma iterativa (sin cargarlos todos en memoria simultáneamente).


In [ ]:
# ── Constantes reutilizadas del notebook de análisis ─────────
STAGE_RAW_TO_LABEL = {
    'SLEEP-S1': 'N1', 'SLEEP-S2': 'N2',
    'SLEEP-S3': 'N3', 'SLEEP-S4': 'N3',
    'SLEEP-REM': 'REM', 'SLEEP-MT': 'MT',
    'WAKE': 'W', 'SLEEP-S0': 'W',
}
STAGE_LABEL_TO_CODE = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'REM': 4, 'MT': 5}
STAGE_CODE_TO_LABEL = {v: k for k, v in STAGE_LABEL_TO_CODE.items()}
STAGE_COLORS = {0:'#E8E8E8', 1:'#AED6F1', 2:'#2980B9', 3:'#1A5276', 4:'#E74C3C', 5:'#F39C12'}

DISORDER_PREFIXES = [
    ('brux','Bruxismo'), ('narco','Narcolepsia'),
    ('nfle','Epilepsia nocturna frontal'), ('plm','Movimientos periódicos'),
    ('rbd','Trastorno REM'), ('sdb','Trastorno respiratorio'), ('ins','Insomnio'),
]

def get_disorder(name):
    for prefix, label in DISORDER_PREFIXES:
        if name.lower().startswith(prefix):
            return label
    return 'Normal' if re.match(r'^n\d', name.lower()) else 'Desconocido'

# ── Parser de anotaciones ────────────────────────────────────
_STAGE_RE = re.compile(r'(SLEEP-S[0-4]|SLEEP-REM|SLEEP-MT|WAKE)\s+(\d+)', re.IGNORECASE)
_CAP_RE   = re.compile(r'(CAP-A[123])\s+(\d+(?:\.\d+)?)', re.IGNORECASE)

def parse_edf_st(filepath):
    patient = filepath.stem.replace('.edf', '')
    text    = filepath.read_bytes().decode('latin-1', errors='replace')
    stage_rows, onset = [], 0.0
    for m in _STAGE_RE.finditer(text):
        raw, dur = m.group(1).upper(), float(m.group(2))
        label = STAGE_RAW_TO_LABEL.get(raw, 'UNKNOWN')
        code  = STAGE_LABEL_TO_CODE.get(label, -1)
        stage_rows.append({
            'patient': patient, 'onset_s': onset, 'duration_s': dur,
            'stage_raw': raw, 'stage': label, 'stage_code': code,
            'stage_label': STAGE_CODE_TO_LABEL.get(code, label),
        })
        onset += dur
    cap_rows, cap_onset = [], 0.0
    for m in _CAP_RE.finditer(text):
        dur = float(m.group(2))
        cap_rows.append({
            'patient': patient, 'onset_s': cap_onset,
            'duration_s': dur, 'subtype': m.group(1).upper(),
        })
        cap_onset += dur
    return pd.DataFrame(stage_rows), pd.DataFrame(cap_rows)

# ── Cargar anotaciones de todos los pacientes ────────────────
st_files = sorted(BASE_PATH.glob('*.edf.st'))
all_stages, all_cap = [], []
for st_path in st_files:
    try:
        s_df, c_df = parse_edf_st(st_path)
        all_stages.append(s_df)
        all_cap.append(c_df)
    except Exception as e:
        print(f'  ⚠️ {st_path.name}: {e}')

stages_all = pd.concat(all_stages, ignore_index=True)
cap_all    = pd.concat(all_cap,    ignore_index=True)
stages_all['disorder'] = stages_all['patient'].map(get_disorder)
cap_all['disorder']    = cap_all['patient'].map(get_disorder)

print(f'Anotaciones cargadas: {stages_all["patient"].nunique()} pacientes')
print(f'  Épocas : {len(stages_all):,}  |  Eventos CAP: {len(cap_all):,}')


In [ ]:
# ── Inventario de archivos EDF ────────────────────────────────
edf_files = sorted(BASE_PATH.glob('*.edf'))
print(f'Total archivos EDF: {len(edf_files)}')

# Leer metadatos sin cargar señales (eficiente en memoria)
edf_meta = []
for f in edf_files:
    try:
        raw = mne.io.read_raw_edf(f, preload=False, verbose=False)
        edf_meta.append({
            'patient':    f.stem,
            'disorder':   get_disorder(f.stem),
            'sfreq':      raw.info['sfreq'],
            'n_channels': len(raw.ch_names),
            'duration_min': raw.times[-1] / 60,
            'channels':   raw.ch_names,
        })
    except Exception as e:
        print(f'  ⚠️ {f.stem}: {e}')

meta_df = pd.DataFrame(edf_meta)
print(f'\nMetadatos leídos: {len(meta_df)} pacientes')
print(f'\nFrecuencias de muestreo:')
print(meta_df['sfreq'].value_counts().sort_index().to_string())


## 4. Exclusión de pacientes n13 y n14

### ¿Qué es la saturación completa?

La **saturación del amplificador** (clipping) ocurre cuando la señal excede el rango dinámico del convertidor analógico-digital (ADC). En la CAP Sleep Database, esto se manifiesta como muestras fijas en ±1000 µV. Una saturación del 100% significa que **todas** las muestras de todos los canales EEG están en el valor máximo, lo que indica un archivo corrupto o un error en la configuración del amplificador durante la adquisición.

### Impacto sobre modelos supervisados

Incluir estos pacientes contaminaría cualquier modelo de ML porque:
- Las características extraídas (potencia espectral, entropía, etc.) serían artefactuales
- El modelo aprendería patrones de ruido, no de actividad cerebral
- Al pertenecer al grupo «Normal», sesgarían la representación de esa clase


In [ ]:
EXCLUIR = ['n13', 'n14']

# Verificar que existen en el dataset
print('Pacientes a excluir:')
for p in EXCLUIR:
    row = meta_df[meta_df['patient'] == p]
    if not row.empty:
        r = row.iloc[0]
        print(f'  ❌ {p}: {r["n_channels"]} canales, {r["sfreq"]:.0f} Hz, {r["disorder"]}')
    else:
        print(f'  ⚠️  {p} no encontrado en metadatos')

# Excluir
n_antes = len(meta_df)
meta_df = meta_df[~meta_df['patient'].isin(EXCLUIR)].reset_index(drop=True)
stages_all = stages_all[~stages_all['patient'].isin(EXCLUIR)].reset_index(drop=True)
cap_all    = cap_all[~cap_all['patient'].isin(EXCLUIR)].reset_index(drop=True)

print(f'\nPacientes antes : {n_antes}')
print(f'Pacientes después: {len(meta_df)}')
print(f'Épocas restantes : {len(stages_all):,}')
print(f'Eventos CAP      : {len(cap_all):,}')


## 5. Normalización de frecuencia de muestreo (resampleo a 256 Hz)

### ¿Qué es la frecuencia de muestreo?

La **frecuencia de muestreo** (sampling rate, *fs*) indica cuántas muestras por segundo captura el ADC. Según el **teorema de Nyquist-Shannon**, para reconstruir fielmente una señal de frecuencia *f*, se necesita *fs* ≥ 2*f*. Si *fs* es insuficiente, se produce **aliasing**: frecuencias altas se «pliegan» sobre frecuencias bajas, generando artefactos irrecuperables.

### ¿Por qué 256 Hz?

La CAP Sleep Database tiene 5 frecuencias distintas (100, 128, 200, 256, 512 Hz). Elegimos 256 Hz porque:

- **Cumple Nyquist para EEG clínico**: la actividad EEG relevante para sueño está por debajo de 100 Hz (delta: 0.5–4 Hz, theta: 4–8, alfa: 8–13, beta: 13–30, gamma: 30–100). Con 256 Hz, la frecuencia de Nyquist es 128 Hz, suficiente.
- **Las guías AASM** recomiendan un mínimo de 200 Hz para EEG de sueño y 256 Hz para señales de mayor frecuencia.
- **Es divisor de 512 Hz** (la frecuencia modal, 75% de los pacientes), lo que simplifica el resampleo.
- **Reduce el volumen de datos** a la mitad para los pacientes a 512 Hz, sin perder información clínica.
- Los pacientes a 100 y 128 Hz se interpolan a 256 Hz. Esto no añade información espectral nueva (las frecuencias por encima de fs/2 original no existían), pero sí homogeneiza la representación temporal.

### Implementación

MNE aplica un filtro antialiasing automático antes del resampleo para evitar aliasing.


In [ ]:
TARGET_SFREQ = 256  # Hz

# Demostración con un paciente a 512 Hz y otro a 256 Hz
demo_patients = ['brux1', 'ins1']  # 512 Hz y 256 Hz respectivamente

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, patient in enumerate(demo_patients):
    edf_path = BASE_PATH / f'{patient}.edf'
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    sfreq_orig = raw.info['sfreq']
    
    # Seleccionar un canal EEG para visualización
    ch_name = raw.ch_names[0]
    
    # Antes del resampleo: 2 segundos de señal
    n_samples = int(2 * sfreq_orig)
    data_orig = raw.get_data(picks=[ch_name])[0][:n_samples] * 1e6  # µV
    time_orig = np.arange(n_samples) / sfreq_orig
    
    # Resamplear
    raw_resampled = raw.copy().resample(TARGET_SFREQ)
    n_samples_new = int(2 * TARGET_SFREQ)
    data_new = raw_resampled.get_data(picks=[ch_name])[0][:n_samples_new] * 1e6
    time_new = np.arange(n_samples_new) / TARGET_SFREQ
    
    # Gráficos
    axes[i, 0].plot(time_orig, data_orig, color='#2980B9', linewidth=0.8)
    axes[i, 0].set_title(f'{patient} — ANTES ({sfreq_orig:.0f} Hz, {len(data_orig)} muestras/2s)', fontweight='bold')
    axes[i, 0].set_xlabel('Tiempo (s)')
    axes[i, 0].set_ylabel('µV')
    
    axes[i, 1].plot(time_new, data_new, color='#E74C3C', linewidth=0.8)
    axes[i, 1].set_title(f'{patient} — DESPUÉS (256 Hz, {len(data_new)} muestras/2s)', fontweight='bold')
    axes[i, 1].set_xlabel('Tiempo (s)')
    axes[i, 1].set_ylabel('µV')
    
    del raw, raw_resampled  # liberar memoria

plt.suptitle('Resampleo a 256 Hz — Comparación antes/después', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'512 Hz → 256 Hz: reduce muestras a la mitad, sin pérdida de información EEG clínica.')
print(f'256 Hz → 256 Hz: sin cambios (ya está en la frecuencia objetivo).')
print(f'100/128 Hz → 256 Hz: interpolación (no añade información espectral nueva).')


## 6. Normalización de nombres de canales

### El problema

La CAP Sleep Database fue recopilada en múltiples centros de sueño, cada uno con su convención de etiquetado. El mismo electrodo aparece con variantes como `C3-A2`, `C3A2`, `C3-P3` o simplemente `C3`. Las guías AASM recomiendan notación como `F3-M2` o `C4-M1`, pero la práctica clínica mantiene variantes históricas.

### Estrategia

Creamos un `CHANNEL_MAPPING` que normaliza cada nombre crudo a una **etiqueta estándar** y una **categoría funcional** (EEG, EMG, ECG, EOG, etc.). Esto permite seleccionar canales homólogos entre pacientes para análisis multicanal.


In [ ]:
CHANNEL_MAPPING = {
    # ── EEG: derivaciones frontales ───────────────────────────
    'FP1-F3': ('Fp1-F3', 'EEG'), 'FP2-F4': ('Fp2-F4', 'EEG'),
    'FP1-A1': ('Fp1',    'EEG'), 'FP2-A2': ('Fp2',    'EEG'),
    'F3-C3':  ('F3-C3',  'EEG'), 'F4-C4':  ('F4-C4',  'EEG'),
    'F3-A1':  ('F3',     'EEG'), 'F3-A2':  ('F3',     'EEG'),
    'F4-A2':  ('F4',     'EEG'), 'F4-A1':  ('F4',     'EEG'),
    'F7-T3':  ('F7-T3',  'EEG'), 'F8-T4':  ('F8-T4',  'EEG'),
    'F3-CZ':  ('F3-Cz',  'EEG'), 'F4-CZ':  ('F4-Cz',  'EEG'),
    'FZ-CZ':  ('Fz-Cz',  'EEG'),
    
    # ── EEG: derivaciones centrales y parietales ──────────────
    'C3-P3':  ('C3-P3',  'EEG'), 'C4-P4':  ('C4-P4',  'EEG'),
    'C3-A2':  ('C3',     'EEG'), 'C3-A1':  ('C3',     'EEG'),
    'C4-A1':  ('C4',     'EEG'), 'C4-A2':  ('C4',     'EEG'),
    'CZ-PZ':  ('Cz-Pz',  'EEG'),
    'P3-O1':  ('P3-O1',  'EEG'), 'P4-O2':  ('P4-O2',  'EEG'),
    'P3-A1':  ('P3',     'EEG'), 'P4-A2':  ('P4',     'EEG'),
    
    # ── EEG: derivaciones occipitales ─────────────────────────
    'O1-A1':  ('O1',     'EEG'), 'O2-A2':  ('O2',     'EEG'),
    
    # ── EEG: derivaciones temporales ──────────────────────────
    'T3-T5':  ('T3-T5',  'EEG'), 'T4-T6':  ('T4-T6',  'EEG'),
    
    # ── EEG: variantes sin guión (multicéntrica) ──────────────
    'F3A2':   ('F3',     'EEG'), 'F4A1':   ('F4',     'EEG'),
    'C3A2':   ('C3',     'EEG'), 'C4A1':   ('C4',     'EEG'),
    'O1A2':   ('O1',     'EEG'), 'O2A1':   ('O2',     'EEG'),
    
    # ── EOG ──────────────────────────────────────────────────
    'ROC-LOC': ('ROC-LOC','EOG'), 'ROC-A1':  ('ROC','EOG'),
    'LOC-A2':  ('LOC',    'EOG'), 'EOG':     ('EOG','EOG'),
    'E1-M2':   ('E1',     'EOG'), 'E2-M2':   ('E2','EOG'),
    'LOC':     ('LOC',    'EOG'), 'ROC':     ('ROC','EOG'),
    
    # ── EMG ──────────────────────────────────────────────────
    'EMG1-EMG2': ('EMG',     'EMG'), 'EMG':      ('EMG', 'EMG'),
    'CHIN1-CHIN2':('Chin',   'EMG'), 'CHIN':     ('Chin','EMG'),
    'CHIN1':     ('Chin1',   'EMG'), 'CHIN2':    ('Chin2','EMG'),
    'LAT1-LAT2': ('LAT',    'EMG'), 'RAT1-RAT2':('RAT', 'EMG'),
    
    # ── ECG ──────────────────────────────────────────────────
    'ECG1-ECG2': ('ECG',    'ECG'), 'ECG':      ('ECG', 'ECG'),
    'ECG2-ECG3': ('ECG',    'ECG'), 'EKG':      ('ECG', 'ECG'),
    
    # ── Respiración ──────────────────────────────────────────
    'FLUSSO':  ('Flow',    'FLOW'), 'FLOW':    ('Flow',   'FLOW'),
    'TORACE':  ('Thorax',  'THORAX'), 'THORAX': ('Thorax', 'THORAX'),
    'ADDOME':  ('Abdomen', 'ABDOMEN'), 'ADDDOME':('Abdomen','ABDOMEN'),
    'ABDOMEN': ('Abdomen', 'ABDOMEN'),
    
    # ── Oximetría ────────────────────────────────────────────
    'SAO2':  ('SpO2', 'OXYGEN'), 'SPO2': ('SpO2', 'OXYGEN'),
    'PLETH': ('Pleth','OXYGEN'),
    
    # ── Otros ────────────────────────────────────────────────
    'POSITION':('Position','OTHER'), 'HR': ('HR','OTHER'),
    'STAT':    ('Status',  'OTHER'),
}

def normalize_channel(ch_name):
    """Devuelve (nombre_normalizado, categoría) para un canal crudo."""
    key = ch_name.strip().upper()
    if key in CHANNEL_MAPPING:
        return CHANNEL_MAPPING[key]
    # Intentar sin espacios
    key_nospace = key.replace(' ', '')
    if key_nospace in CHANNEL_MAPPING:
        return CHANNEL_MAPPING[key_nospace]
    return (ch_name, 'UNKNOWN')

# ── Demostración ─────────────────────────────────────────────
demo_path = BASE_PATH / 'nfle1.edf'
raw_demo = mne.io.read_raw_edf(demo_path, preload=False, verbose=False)

print(f'Canales originales de nfle1 ({len(raw_demo.ch_names)}):')
print(f'  {raw_demo.ch_names}')
print()
print(f'{"Original":<20} {"Normalizado":<15} {"Categoría":<10}')
print('-' * 45)
for ch in raw_demo.ch_names:
    norm_name, cat = normalize_channel(ch)
    flag = ' ⚠️' if cat == 'UNKNOWN' else ''
    print(f'{ch:<20} {norm_name:<15} {cat:<10}{flag}')

del raw_demo


## 7. Limpieza y selección de canales

### Estrategia de selección

Para análisis multimodal (EEG + EMG + ECG) necesitamos identificar qué canales son relevantes y cuáles son ruido o metadatos codificados como señal.

**Se conservan:**
- Canales **EEG**: toda la actividad cerebral
- Canales **EMG**: actividad muscular (importante para estadificación de REM)
- Canales **ECG**: frecuencia cardíaca (relevante para detección de apneas)
- Canales **EOG**: movimientos oculares (diferencia entre REM y NREM)

**Se excluyen:**
- `POSITION`: codifica posición corporal como valor discreto, no es señal fisiológica
- `HR`, `STAT`: metadatos, no señales de amplificador
- `SAO2`, `SPO2`, `PLETH`: señales de rango fijo que saturan trivialmente al umbral EEG
- Canales `UNKNOWN` que no pudieron mapearse


In [ ]:
KEEP_CATEGORIES = {'EEG', 'EMG', 'ECG', 'EOG'}

def select_channels(raw):
    """
    Selecciona canales relevantes de un Raw de MNE.
    Devuelve la lista de nombres originales a conservar y sus categorías.
    """
    keep = []
    info = []
    for ch in raw.ch_names:
        norm_name, cat = normalize_channel(ch)
        if cat in KEEP_CATEGORIES:
            keep.append(ch)
            info.append({'original': ch, 'normalizado': norm_name, 'categoria': cat})
    return keep, pd.DataFrame(info)

# ── Demostración con varios pacientes ────────────────────────
demo_patients = ['brux1', 'nfle1', 'n1', 'sdb1']

for patient in demo_patients:
    edf_path = BASE_PATH / f'{patient}.edf'
    if not edf_path.exists():
        continue
    raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
    keep, info_df = select_channels(raw)
    excluded = [ch for ch in raw.ch_names if ch not in keep]
    
    print(f'\n── {patient} ({get_disorder(patient)}) ──')
    print(f'  Total canales  : {len(raw.ch_names)}')
    print(f'  Conservados    : {len(keep)} ({info_df["categoria"].value_counts().to_dict()})')
    print(f'  Excluidos ({len(excluded)}): {excluded}')
    del raw


## 8. Escalamiento por canal (StandardScaler)

### ¿Por qué escalar señales EEG?

Las señales EEG tienen amplitudes típicas de ±50–200 µV, pero los canales EMG pueden alcanzar ±500 µV y el ECG ±1000 µV. Si alimentamos un modelo de ML con señales sin escalar, los canales de mayor amplitud dominarán el aprendizaje simplemente por su magnitud, no por su contenido informativo.

### ¿Por qué StandardScaler?

`StandardScaler` transforma cada canal a media 0 y desviación estándar 1:

$$z = \frac{x - \mu}{\sigma}$$

Es preferible a `MinMaxScaler` para EEG porque:
- Las señales EEG son **aproximadamente gaussianas**, por lo que la estandarización preserva su distribución natural
- `MinMaxScaler` colapsa el rango útil si hay outliers de saturación (±1000 µV comprimen toda la señal al rango [0.001, 0.999])
- `RobustScaler` es una alternativa válida si la saturación es frecuente, ya que usa mediana e IQR en lugar de media y DE

### Implementación

El escalamiento se aplica **por canal y por paciente** (cada canal se normaliza independientemente).


In [ ]:
# ── Demostración visual: antes vs después ────────────────────
patient_sc = 'brux1'
edf_path = BASE_PATH / f'{patient_sc}.edf'
raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

keep_chs, _ = select_channels(raw)
raw.pick(keep_chs)

# Tomar 5 minutos de señal
n_5min = int(5 * 60 * raw.info['sfreq'])
data_raw = raw.get_data()[:, :n_5min] * 1e6  # µV
ch_names = raw.ch_names[:6]  # primeros 6 canales para visualización

# Aplicar escaladores
scaler_std    = StandardScaler()
scaler_robust = RobustScaler()

data_std    = scaler_std.fit_transform(data_raw[:6].T).T
data_robust = scaler_robust.fit_transform(data_raw[:6].T).T

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
time_s = np.arange(data_raw.shape[1]) / raw.info['sfreq']

# Mostrar primeros 30 segundos
n_30s = int(30 * raw.info['sfreq'])

for j, ch in enumerate(ch_names[:4]):
    axes[0].plot(time_s[:n_30s], data_raw[j, :n_30s], label=ch, alpha=0.7, linewidth=0.6)
    axes[1].plot(time_s[:n_30s], data_std[j, :n_30s], label=ch, alpha=0.7, linewidth=0.6)
    axes[2].plot(time_s[:n_30s], data_robust[j, :n_30s], label=ch, alpha=0.7, linewidth=0.6)

axes[0].set_title('Original (µV)', fontweight='bold')
axes[0].legend(fontsize=8, loc='upper right')
axes[1].set_title('StandardScaler (z-score)', fontweight='bold')
axes[1].legend(fontsize=8, loc='upper right')
axes[2].set_title('RobustScaler (mediana/IQR)', fontweight='bold')
axes[2].set_xlabel('Tiempo (s)')
axes[2].legend(fontsize=8, loc='upper right')

plt.suptitle(f'Comparación de escaladores — {patient_sc} (30 s)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

del raw


In [ ]:
# ── Comparación estadística y distribuciones ─────────────────
patient_sc = 'brux1'
raw = mne.io.read_raw_edf(BASE_PATH / f'{patient_sc}.edf', preload=True, verbose=False)
keep_chs, _ = select_channels(raw)
raw.pick(keep_chs)

n_5min = int(5 * 60 * raw.info['sfreq'])
data_raw = raw.get_data()[:, :n_5min] * 1e6

ch_idx = 0
ch_name = raw.ch_names[ch_idx]

original = data_raw[ch_idx]
scaled_std = StandardScaler().fit_transform(original.reshape(-1, 1)).flatten()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histograma antes
axes[0].hist(original, bins=100, color='#2980B9', edgecolor='white', alpha=0.8, density=True)
axes[0].set_title(f'Original — {ch_name}', fontweight='bold')
axes[0].set_xlabel('µV')

# Histograma después
axes[1].hist(scaled_std, bins=100, color='#E74C3C', edgecolor='white', alpha=0.8, density=True)
axes[1].set_title(f'StandardScaler — {ch_name}', fontweight='bold')
axes[1].set_xlabel('z-score')

# Boxplot comparativo
bp = axes[2].boxplot(
    [original, scaled_std],
    labels=['Original\n(µV)', 'Escalado\n(z-score)'],
    patch_artist=True,
    medianprops={'color': 'black', 'linewidth': 2},
    flierprops={'marker': '.', 'markersize': 1, 'alpha': 0.2},
)
bp['boxes'][0].set_facecolor('#2980B9')
bp['boxes'][1].set_facecolor('#E74C3C')
axes[2].set_title('Comparación de distribuciones', fontweight='bold')

plt.suptitle(f'Efecto del escalamiento — {patient_sc}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Canal: {ch_name}')
print(f'  Original     → Media: {original.mean():.2f}, DE: {original.std():.2f}, Min: {original.min():.1f}, Max: {original.max():.1f}')
print(f'  StandardScaler → Media: {scaled_std.mean():.4f}, DE: {scaled_std.std():.4f}, Min: {scaled_std.min():.2f}, Max: {scaled_std.max():.2f}')

del raw


## 9. Normalización por tiempo de sueño efectivo

### ¿Qué es el tiempo de sueño efectivo?

El **tiempo total de grabación** (TRT, Total Recording Time) incluye los períodos de vigilia previos al inicio del sueño, los despertares nocturnos y la vigilia terminal. El **tiempo de sueño efectivo** (TST, Total Sleep Time) es la suma del tiempo en N1, N2, N3 y REM, excluyendo W y MT.

### ¿Por qué normalizar?

Si contamos «eventos CAP por paciente» sin normalizar, un paciente con 10 horas de grabación tendrá más eventos que uno con 4 horas, independientemente de la severidad de su trastorno. La métrica correcta es **tasa CAP** (eventos por hora de sueño efectivo), que es comparable entre pacientes.


In [ ]:
# ── Calcular TST y TRT por paciente ───────────────────────────
sleep_stages = {'N1', 'N2', 'N3', 'REM'}

tst_by_patient = (
    stages_all[stages_all['stage_label'].isin(sleep_stages)]
    .groupby('patient')['duration_s'].sum()
    .rename('tst_s')
)
trt_by_patient = (
    stages_all.groupby('patient')['duration_s'].sum()
    .rename('trt_s')
)
cap_by_patient = (
    cap_all.groupby('patient')['subtype'].count()
    .rename('n_cap')
)

sleep_metrics = pd.DataFrame({
    'trt_s':  trt_by_patient,
    'tst_s':  tst_by_patient,
    'n_cap':  cap_by_patient,
}).fillna(0)

sleep_metrics['trt_h'] = sleep_metrics['trt_s'] / 3600
sleep_metrics['tst_h'] = sleep_metrics['tst_s'] / 3600
sleep_metrics['eficiencia_pct'] = (sleep_metrics['tst_s'] / sleep_metrics['trt_s'] * 100).round(1)
sleep_metrics['cap_rate_trt'] = (sleep_metrics['n_cap'] / sleep_metrics['trt_h']).round(1)
sleep_metrics['cap_rate_tst'] = (sleep_metrics['n_cap'] / sleep_metrics['tst_h']).round(1)
sleep_metrics['disorder'] = sleep_metrics.index.map(get_disorder)

print('Métricas de sueño (primeros 15 pacientes):')
display(sleep_metrics[['disorder','trt_h','tst_h','eficiencia_pct','n_cap','cap_rate_trt','cap_rate_tst']].head(15))

print(f'\nEficiencia de sueño promedio: {sleep_metrics["eficiencia_pct"].mean():.1f}%')
print(f'Tasa CAP (por hora TRT): {sleep_metrics["cap_rate_trt"].mean():.1f} eventos/h')
print(f'Tasa CAP (por hora TST): {sleep_metrics["cap_rate_tst"].mean():.1f} eventos/h')


In [ ]:
# ── Comparación: tasa CAP por TRT vs TST por trastorno ───────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes,
    ['cap_rate_trt', 'cap_rate_tst'],
    ['Tasa CAP normalizada por TRT\n(eventos / hora total)', 
     'Tasa CAP normalizada por TST\n(eventos / hora de sueño efectivo)']):
    
    order = sleep_metrics.groupby('disorder')[col].median().sort_values(ascending=False).index
    sns.boxplot(data=sleep_metrics, x='disorder', y=col, order=order,
                palette='Set2', ax=ax, fliersize=3)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Eventos / hora')
    ax.tick_params(axis='x', rotation=35)

plt.suptitle('Impacto de la normalización temporal en métricas CAP', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Observación: la tasa por TST es más alta que por TRT porque excluye la vigilia.')
print('La diferencia es especialmente marcada en Insomnio (40% de vigilia).')


## 10. Balanceo de clases

### El problema del desbalance en medicina

En la CAP Sleep Database, NFLE tiene 40 pacientes (37%) mientras que Bruxismo tiene solo 2 (1.9%). Un clasificador que prediga siempre «NFLE» tendría 37% de accuracy sin aprender nada útil. Esto es especialmente peligroso en medicina, donde la clase minoritaria suele ser la más clínicamente relevante.

### Estrategias evaluadas

| Estrategia | Ventaja | Riesgo |
|---|---|---|
| **SMOTE** | Genera muestras sintéticas de la clase minoritaria | Puede crear muestras irreales si el espacio de características es ruidoso |
| **Ponderación de clases** | No modifica los datos, ajusta la función de pérdida | Menos efectiva con desbalance extremo |
| **Undersampling** | Reduce la clase mayoritaria | Pierde información valiosa |

### ⚠️ Riesgos de aplicar SMOTE incorrectamente

SMOTE debe aplicarse **después** de la extracción de características y **solo sobre el conjunto de entrenamiento**:
- Aplicarlo antes de extraer características genera muestras en el espacio temporal que pueden no tener sentido fisiológico
- Aplicarlo sobre todo el dataset (incluyendo test) produce **data leakage**: el modelo ve muestras sintéticas derivadas de datos de test durante el entrenamiento, inflando artificialmente las métricas

### Demostración

Aquí demostramos las estrategias sobre un dataset simplificado de características por paciente (no sobre las señales crudas).


In [ ]:
# ── Construir dataset de características por paciente ─────────
# Usamos las métricas de sueño como proxy de características
feature_df = sleep_metrics[['tst_h', 'eficiencia_pct', 'n_cap', 'cap_rate_tst', 'disorder']].copy()
feature_df = feature_df[feature_df['disorder'] != 'Desconocido'].dropna()

print('Distribución original de clases:')
class_counts = feature_df['disorder'].value_counts()
print(class_counts.to_string())
print(f'\nTotal pacientes: {len(feature_df)}')
print(f'Ratio max/min: {class_counts.max()}:{class_counts.min()}')


In [ ]:
# ── Ponderación de clases ─────────────────────────────────────
classes = feature_df['disorder'].unique()
y = feature_df['disorder'].values

weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
weight_dict = dict(zip(np.unique(y), weights))

print('Pesos calculados por clase (sklearn compute_class_weight):')
print(f'{"Clase":<35} {"N":>5} {"Peso":>8}')
print('-' * 50)
for cls in sorted(weight_dict, key=weight_dict.get, reverse=True):
    n = class_counts.get(cls, 0)
    print(f'{cls:<35} {n:>5} {weight_dict[cls]:>8.2f}')

print()
print('Interpretación: la clase Bruxismo (n=2) recibe peso 6.63×,')
print('mientras NFLE (n=40) recibe peso 0.33×. Esto equilibra la')
print('contribución de cada clase en la función de pérdida.')


In [ ]:
# ── SMOTE (si disponible) ─────────────────────────────────────
if SMOTE_AVAILABLE:
    X = feature_df[['tst_h', 'eficiencia_pct', 'n_cap', 'cap_rate_tst']].values
    y = feature_df['disorder'].values
    
    # SMOTE necesita al menos k_neighbors+1 muestras por clase
    min_class_size = min(class_counts.values)
    k = min(5, min_class_size - 1)  # adaptar k_neighbors
    
    if k >= 1:
        smote = SMOTE(k_neighbors=k, random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)
        
        resampled_counts = pd.Series(y_resampled).value_counts()
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        class_counts.sort_values().plot.barh(ax=axes[0], color='#2980B9', edgecolor='white')
        axes[0].set_title('Distribución ANTES de SMOTE', fontweight='bold')
        axes[0].set_xlabel('N° de pacientes')
        for i, (val, name) in enumerate(zip(class_counts.sort_values().values, class_counts.sort_values().index)):
            axes[0].text(val + 0.2, i, str(val), va='center', fontweight='bold', fontsize=9)
        
        resampled_counts.sort_values().plot.barh(ax=axes[1], color='#27AE60', edgecolor='white')
        axes[1].set_title('Distribución DESPUÉS de SMOTE', fontweight='bold')
        axes[1].set_xlabel('N° de muestras')
        for i, (val, name) in enumerate(zip(resampled_counts.sort_values().values, resampled_counts.sort_values().index)):
            axes[1].text(val + 0.2, i, str(val), va='center', fontweight='bold', fontsize=9)
        
        plt.suptitle('Efecto de SMOTE sobre el balance de clases', fontsize=13, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
        
        print(f'Muestras originales : {len(X)}')
        print(f'Muestras con SMOTE  : {len(X_resampled)}')
        print(f'k_neighbors usado   : {k}')
    else:
        print('⚠️  Clases con menos de 2 muestras. SMOTE no aplicable.')
        print('   Se recomienda usar solo ponderación de clases.')
else:
    print('SMOTE no disponible. Instalar: pip install imbalanced-learn')
    print('Se recomienda usar ponderación de clases como alternativa.')


## 11. Generación de datasets preprocesados

Aplicamos el pipeline completo a un subconjunto de pacientes y guardamos los resultados listos para extracción de características. El pipeline por paciente es:

1. Cargar EDF
2. Excluir si es n13/n14
3. Seleccionar canales (EEG + EMG + ECG + EOG)
4. Resamplear a 256 Hz
5. Aplicar StandardScaler por canal
6. Guardar


In [ ]:
def preprocess_patient(patient_name, base_path, target_sfreq=256):
    """
    Pipeline completo de preprocesamiento para un paciente.
    Retorna (data_scaled, ch_names, ch_categories, sfreq) o None si falla.
    """
    edf_path = base_path / f'{patient_name}.edf'
    if not edf_path.exists():
        return None
    
    try:
        # 1. Cargar
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        
        # 2. Seleccionar canales
        keep_chs, info_df = select_channels(raw)
        if len(keep_chs) == 0:
            print(f'  ⚠️  {patient_name}: sin canales válidos')
            return None
        raw.pick(keep_chs)
        
        # 3. Resamplear
        if raw.info['sfreq'] != target_sfreq:
            raw.resample(target_sfreq)
        
        # 4. Obtener datos en µV
        data = raw.get_data() * 1e6  # V → µV
        
        # 5. Escalar por canal
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(data.T).T  # (n_channels, n_samples)
        
        # Info de canales
        ch_categories = [normalize_channel(ch)[1] for ch in keep_chs]
        
        return {
            'data': data_scaled,
            'ch_names': list(raw.ch_names),
            'ch_categories': ch_categories,
            'sfreq': target_sfreq,
            'n_samples': data_scaled.shape[1],
            'duration_min': data_scaled.shape[1] / target_sfreq / 60,
        }
    except Exception as e:
        print(f'  ❌ {patient_name}: {e}')
        return None
    finally:
        del raw

# ── Procesar pacientes de muestra ────────────────────────────
sample_patients = ['brux1', 'ins1', 'narco1', 'nfle1', 'plm1', 'rbd1', 'sdb1', 'n1']
results = {}

print(f'{"Paciente":<10} {"Canales":>8} {"Muestras":>12} {"Duración":>10} {"Categorías"}')
print('-' * 70)

for patient in sample_patients:
    result = preprocess_patient(patient, BASE_PATH)
    if result:
        results[patient] = result
        cats = pd.Series(result['ch_categories']).value_counts().to_dict()
        print(f'{patient:<10} {len(result["ch_names"]):>8} {result["n_samples"]:>12,} {result["duration_min"]:>8.1f} min  {cats}')

print(f'\nPacientes preprocesados: {len(results)} / {len(sample_patients)}')


In [ ]:
# ── Guardar datasets preprocesados ────────────────────────────
for patient, result in results.items():
    out_path = PROCESSED_DIR / f'{patient}_preprocessed.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump({
            'patient':       patient,
            'disorder':      get_disorder(patient),
            'data':          result['data'],
            'ch_names':      result['ch_names'],
            'ch_categories': result['ch_categories'],
            'sfreq':         result['sfreq'],
        }, f)

# Guardar métricas de sueño
sleep_metrics.to_csv(PROCESSED_DIR / 'sleep_metrics.csv')

# Guardar pesos de clases
pd.Series(weight_dict).to_csv(PROCESSED_DIR / 'class_weights.csv')

print(f'Archivos guardados en: {PROCESSED_DIR.resolve()}')
print()
for f in sorted(PROCESSED_DIR.glob('*')):
    size = f.stat().st_size
    if size > 1e6:
        print(f'  {f.name:<40} {size/1e6:.1f} MB')
    else:
        print(f'  {f.name:<40} {size/1e3:.1f} KB')


## 12. Visualizaciones de validación

Verificamos que el preprocesamiento produjo señales coherentes comparando un paciente antes y después del pipeline.


In [ ]:
# ── Comparación completa: antes vs después del pipeline ──────
patient_viz = 'brux1'
edf_path = BASE_PATH / f'{patient_viz}.edf'
raw_orig = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
result_proc = results[patient_viz]

# 30 segundos de señal
n_orig = int(30 * raw_orig.info['sfreq'])
n_proc = int(30 * result_proc['sfreq'])

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)

# Señal original (primeros 4 canales)
for j in range(min(4, len(raw_orig.ch_names))):
    data_ch = raw_orig.get_data()[j, :n_orig] * 1e6
    time_ch = np.arange(n_orig) / raw_orig.info['sfreq']
    axes[0].plot(time_ch, data_ch + j * 100, label=raw_orig.ch_names[j], linewidth=0.5)

axes[0].set_title(f'{patient_viz} — ANTES del preprocesamiento ({raw_orig.info["sfreq"]:.0f} Hz, {len(raw_orig.ch_names)} canales)', fontweight='bold')
axes[0].set_ylabel('µV (offset para visualización)')
axes[0].legend(fontsize=7, loc='upper right')

# Señal preprocesada (primeros 4 canales)
for j in range(min(4, len(result_proc['ch_names']))):
    data_ch = result_proc['data'][j, :n_proc]
    time_ch = np.arange(n_proc) / result_proc['sfreq']
    axes[1].plot(time_ch, data_ch + j * 5, label=result_proc['ch_names'][j], linewidth=0.5)

axes[1].set_title(f'{patient_viz} — DESPUÉS del preprocesamiento (256 Hz, {len(result_proc["ch_names"])} canales, StandardScaler)', fontweight='bold')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('z-score (offset para visualización)')
axes[1].legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.show()

del raw_orig


## 13. Análisis final y conclusiones


In [ ]:
print('='*65)
print('  RESUMEN DEL PREPROCESAMIENTO — CAP Sleep Database')
print('='*65)

print(f'''
⚙️  PIPELINE APLICADO
  1. Exclusión de pacientes    : n13, n14 (saturación EEG 100%)
  2. Resampleo                 : todas las señales a 256 Hz
  3. Normalización de canales  : CHANNEL_MAPPING con {len(CHANNEL_MAPPING)} entradas
  4. Selección de canales      : EEG + EMG + ECG + EOG
  5. Escalamiento              : StandardScaler por canal (z-score)
  6. Normalización temporal    : métricas por hora de sueño efectivo (TST)
  7. Balanceo de clases        : ponderación {'+ SMOTE' if SMOTE_AVAILABLE else '(SMOTE no disponible)'}

📊 RESULTADO
  Pacientes procesados         : {len(results)}
  Frecuencia uniforme          : 256 Hz
  Canales conservados          : EEG, EMG, ECG, EOG
  Canales excluidos            : Position, HR, SAO2, SPO2, PLETH, STAT
  Escalamiento                 : media ≈ 0, DE ≈ 1 por canal

📁 ARCHIVOS GENERADOS
  Directorio: {PROCESSED_DIR.resolve()}
  - {{paciente}}_preprocessed.pkl : señales preprocesadas + metadatos
  - sleep_metrics.csv            : métricas de sueño por paciente
  - class_weights.csv            : pesos para clasificación balanceada

✅ IMPACTO ESPERADO EN CLASIFICACIÓN
  • Señales homogéneas a 256 Hz → características espectrales comparables
  • StandardScaler → canales con contribución equitativa al modelo
  • Ponderación de clases → el clasificador no se sesga hacia NFLE
  • TST como denominador → métricas CAP comparables entre pacientes

⚠️  LIMITACIONES
  • El resampleo de 100/128 Hz a 256 Hz no añade información espectral
    por encima del Nyquist original (50/64 Hz)
  • StandardScaler asume estacionariedad de la señal, que es aproximada
    en ventanas cortas pero no en grabaciones completas de 8+ horas
  • SMOTE sobre pocas muestras (Bruxismo: n=2) puede generar sintéticos
    poco representativos; la ponderación es más segura en este caso
  • No se aplicó filtrado de artefactos (ICA, ASR) en esta etapa;
    puede añadirse como paso intermedio si las métricas lo requieren

🔄 PRÓXIMOS PASOS
  1. Segmentar las señales en épocas de 30 s alineadas al hipnograma
  2. Extraer características por época (potencia por banda, entropía,
     hjorth, coherencia, etc.)
  3. Entrenar clasificadores (Random Forest, SVM, XGBoost, redes neuronales)
  4. Evaluar con validación cruzada estratificada y pesos de clase
''')
